In [1]:
%env HF_ENDPOINT=https://hf-mirror.com

env: HF_ENDPOINT=https://hf-mirror.com


In [3]:
# 加载模型与TOkenizer
from transformers import AutoModelForCausalLM,AutoTokenizer
import torch
from transformers import BitsAndBytesConfig

# 量化配置
config=BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16
)

model_name = "Qwen/Qwen3-4B"
model = AutoModelForCausalLM.from_pretrained(model_name,quantization_config=config)
tokenizer = AutoTokenizer.from_pretrained(model_name)

Loading weights: 100%|██████████████████████████████████████████████████████████████████████████████████████| 398/398 [00:03<00:00, 131.52it/s]


In [4]:
from peft import prepare_model_for_kbit_training
model = prepare_model_for_kbit_training(model)


In [5]:
# 数据集 处理数据集至openai格式
from datasets import load_dataset
dataset_dict = load_dataset("json",data_files={"train":"data/keywords_data_train.jsonl",
                                              "test":"data/keywords_data_test.jsonl"})
# 转成openai格式
def map_func(exapmle):
    conversation = exapmle["conversation"]
    messages=[]
    for item in conversation:
        messages.append({"role":"user","content":item["human"]})
        messages.append({"role":"assistant","content":item["assistant"]})
    return {"messages":messages}

dataset_dict=dataset_dict.map(map_func,batched=False,remove_columns=["conversation_id","category","conversation","dataset"])

In [6]:
from peft import LoraConfig
from trl import SFTConfig,SFTTrainer
# TODO: Configure LoRA parameters
# r: rank dimension for LoRA update matrices (smaller = more compression)
rank_dimension = 4
# lora_alpha: scaling factor for LoRA layers (higher = stronger adaptation)
lora_alpha = 8
# lora_dropout: dropout probability for LoRA layers (helps prevent overfitting)
lora_dropout = 0.05

training_args = SFTConfig(
    output_dir="/home/tianjp/llmLearn/stf/Qwen3-4B/Qlora",
    max_steps=1000,
    per_device_train_batch_size=2,
    learning_rate=5e-5,
    logging_steps=10,
    save_total_limit=2,
    save_steps=100,
    eval_strategy="steps",
    eval_steps=100,
    load_best_model_at_end=True,
    bf16=True,
    warmup_steps=50,
    assistant_only_loss=True,
)


peft_config = LoraConfig(
    r=rank_dimension,  # Rank dimension - typically between 4-32
    lora_alpha=lora_alpha,  # LoRA scaling factor - typically 2x rank
    lora_dropout=lora_dropout,  # Dropout probability for LoRA layers
    bias="none",  # Bias type for LoRA. the corresponding biases will be updated during training.
    target_modules="all-linear",  # Which modules to apply LoRA to
    task_type="CAUSAL_LM",  # Task type for model architecture
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset_dict["train"],
    eval_dataset=dataset_dict["test"],
    peft_config=peft_config,  # LoRA configuration
    processing_class=tokenizer,
)



In [7]:
dataloader = trainer.get_train_dataloader()
batch=next(iter(dataloader))
batch["input_ids"].shape

torch.Size([2, 166])

In [8]:
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss,Validation Loss
100,0.880328,1.209091
200,0.809243,1.143347
300,1.048299,1.119371
400,1.111384,1.113039
500,0.813219,1.097752
600,1.171883,1.091713
700,0.808407,1.090232
800,1.012376,1.086835
900,1.005565,1.084966
1000,1.294249,1.084271


/home/tianjp/anaconda3/envs/llm/lib/python3.12/site-packages/peft/utils/other.py:1419: UserWarning: Unable to fetch remote file due to the following error [Errno 101] Network is unreachable - silently ignoring the lookup for the file config.json in Qwen/Qwen3-4B.
  warnings.warn(
/home/tianjp/anaconda3/envs/llm/lib/python3.12/site-packages/peft/utils/save_and_load.py:372: UserWarning: Could not find a config file in Qwen/Qwen3-4B - will assume that the vocabulary was not modified.
  warnings.warn(
/home/tianjp/anaconda3/envs/llm/lib/python3.12/site-packages/peft/utils/other.py:1419: UserWarning: Unable to fetch remote file due to the following error [Errno 101] Network is unreachable - silently ignoring the lookup for the file config.json in Qwen/Qwen3-4B.
  warnings.warn(
/home/tianjp/anaconda3/envs/llm/lib/python3.12/site-packages/peft/utils/save_and_load.py:372: UserWarning: Could not find a config file in Qwen/Qwen3-4B - will assume that the vocabulary was not modified.
  warnings.

TrainOutput(global_step=1000, training_loss=1.2554393224716187, metrics={'train_runtime': 1155.65, 'train_samples_per_second': 1.731, 'train_steps_per_second': 0.865, 'total_flos': 9609682776545280.0, 'train_loss': 1.2554393224716187})

In [9]:
trainer.save_model("/home/tianjp/llmLearn/stf/Qwen3-4B/Qlora/best")

/home/tianjp/anaconda3/envs/llm/lib/python3.12/site-packages/peft/utils/other.py:1419: UserWarning: Unable to fetch remote file due to the following error [Errno 101] Network is unreachable - silently ignoring the lookup for the file config.json in Qwen/Qwen3-4B.
  warnings.warn(
/home/tianjp/anaconda3/envs/llm/lib/python3.12/site-packages/peft/utils/save_and_load.py:372: UserWarning: Could not find a config file in Qwen/Qwen3-4B - will assume that the vocabulary was not modified.
  warnings.warn(
